# Notebook 04 — Clustering: Crime Hotspot Detection (Task 2)

Applies K-Means and DBSCAN to Chicago crime coordinates to detect geographic hotspots.

**Prerequisites:** `data/samples/chicago_sample.csv` must exist.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.clustering import (
    run_kmeans, run_dbscan, analyze_clusters, elbow_data, tune_dbscan
)
from src.evaluation.metrics import (
    plot_elbow_curve, plot_cluster_scatter
)
from src.visualization.maps import make_heatmap, make_cluster_map, make_arrest_rate_map

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/reports', exist_ok=True)

## 1. Load sample data

In [ ]:
df = pd.read_csv('../data/samples/chicago_sample.csv', low_memory=False)
df['Arrest'] = pd.to_numeric(df['Arrest'], errors='coerce').fillna(0).astype(int)
df = df.dropna(subset=['Latitude', 'Longitude'])
print(f'Loaded {len(df):,} rows with valid coordinates')
df[['Latitude', 'Longitude']].describe()

## 2. K-Means clustering

In [ ]:
best_km, all_km = run_kmeans(df, k_values=[5, 8, 10, 12, 15])

In [ ]:
# Elbow curve
elbow_df = elbow_data(all_km)
plot_elbow_curve(elbow_df, output_dir='../outputs/figures')
elbow_df

In [ ]:
# Cluster analysis
df_km, km_stats = analyze_clusters(df, best_km['labels'], name='KMeans')
km_stats.to_csv('../outputs/reports/kmeans_cluster_stats.csv', index=False)
km_stats

In [ ]:
# Scatter plot
plot_cluster_scatter(
    df_km, title=f'K-Means Clusters (k={best_km["k"]})',
    output_dir='../outputs/figures', filename='kmeans_scatter.png'
)

In [ ]:
# Cluster arrest rates
plt.figure(figsize=(10, 4))
km_stats_sorted = km_stats.sort_values('arrest_rate')
plt.barh(km_stats_sorted['Cluster'].astype(str), km_stats_sorted['arrest_rate'],
         color='steelblue')
plt.axvline(df['Arrest'].mean(), ls='--', color='red', label='Overall mean')
plt.title('Arrest Rate per K-Means Cluster')
plt.xlabel('Arrest Rate')
plt.ylabel('Cluster ID')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/kmeans_arrest_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. DBSCAN clustering

In [ ]:
# DBSCAN parameter tuning
dbscan_grid = tune_dbscan(
    df,
    eps_values=[0.003, 0.005, 0.008, 0.01],
    min_samples_values=[30, 50, 100]
)
dbscan_grid.to_csv('../outputs/reports/dbscan_grid_search.csv', index=False)
dbscan_grid

In [ ]:
# Run DBSCAN with best parameters
db_labels, n_clusters = run_dbscan(df, eps=0.005, min_samples=50)

In [ ]:
df_db, db_stats = analyze_clusters(df, db_labels, name='DBSCAN')
db_stats.to_csv('../outputs/reports/dbscan_cluster_stats.csv', index=False)
db_stats.head(10)

In [ ]:
plot_cluster_scatter(
    df_db, title='DBSCAN Clusters',
    output_dir='../outputs/figures', filename='dbscan_scatter.png'
)

## 4. K-Means vs DBSCAN comparison

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler

coords = df[['Latitude', 'Longitude']].values
coords_scaled = StandardScaler().fit_transform(coords)

# K-Means scores
km_sil = silhouette_score(coords_scaled, best_km['labels'],
                          sample_size=10000, random_state=42)
km_db  = davies_bouldin_score(coords_scaled, best_km['labels'])

# DBSCAN scores (non-noise)
mask = db_labels != -1
db_sil = silhouette_score(coords[mask], db_labels[mask],
                          sample_size=min(10000, mask.sum()), random_state=42) if mask.sum() > 1 else None

comparison = pd.DataFrame({
    'Algorithm':       ['K-Means', 'DBSCAN'],
    'n_clusters':      [best_km['k'], n_clusters],
    'silhouette':      [round(km_sil, 4), round(db_sil, 4) if db_sil else 'N/A'],
    'davies_bouldin':  [round(km_db, 4), 'N/A'],
})
print('Algorithm Comparison:')
print(comparison.to_string(index=False))
comparison.to_csv('../outputs/reports/clustering_comparison.csv', index=False)

## 5. Interactive Folium maps

In [ ]:
# Crime heatmap (all crimes)
m_heat = make_heatmap(df, output_path='../outputs/figures/crime_heatmap.html')
m_heat

In [ ]:
# Arrest-rate heatmap
m_arrest = make_arrest_rate_map(df, output_path='../outputs/figures/arrest_rate_map.html')
m_arrest

In [ ]:
# K-Means cluster map
m_cluster = make_cluster_map(
    df_km, km_stats,
    output_path='../outputs/figures/kmeans_cluster_map.html'
)
m_cluster

## 6. Top hotspot districts

In [ ]:
if 'District' in df_km.columns:
    top_districts = (
        df_km[df_km['Cluster'] != -1]
        .groupby('District')
        .agg(crime_count=('Arrest', 'count'), arrest_rate=('Arrest', 'mean'))
        .sort_values('crime_count', ascending=False)
        .head(10)
    )
    print('Top 10 Districts by Crime Count:')
    print(top_districts.to_string())